In [7]:
# ORM_Example.ipynb
# !uv pip install ipykernel

# Django Shell을 주피터랩에서 실행할 수 있도록 환경설정
import os
import django
os.environ['DJANGO_SETTINGS_MODULE'] = "config.settings"
os.environ['DJANGO_ALLOW_ASYNC_UNSAFE'] = "true"

django.setup()

In [2]:
from polls.models import Question, Choice

# ModelClass.objects : (Model)Manager객체 -> select 처리 메소드들을 제공.
type(Question.objects)

django.db.models.manager.Manager

In [3]:
# pk=1 인 질문 조회
result = Question.objects.get(pk=1)
print(type(result))

<class 'polls.models.Question'>


In [4]:
result

<Question: 1. 좋아하는 색은 무엇입니까?>

In [5]:
result.pk, result.question_text, result.pub_date

(1,
 '좋아하는 색은 무엇입니까?',
 datetime.datetime(2026, 7, 24, 6, 39, 36, 233670, tzinfo=datetime.timezone.utc))

# 조회
- Model Manager의 메소드들을 이용해서 조회
- `all()`: 전체 조회
- (where 절)조건으로 조회
    - `filter()`: 조건이 true 행들을 조회
    - `exclude()`: 조건이 false 행들을 조회
    - `get()`: 조건이 true 한개 행을 조회.(조회결과가 하나일때 사용.)
- all, filter, exclude, get 반환타입: `QuerySet`
- get 반환: 모델 객체

In [4]:
# 모든 질문들을 조회
result = Question.objects.all()
print(type(result)) 
print("결과개수:", len(result))
print("첫번째 조회결과:", result[0], type(result[0]))
print(result.query) # sql문조회

<class 'django.db.models.query.QuerySet'>
결과개수: 4
첫번째 조회결과: 1. 좋아하는 색은 무엇입니까? <class 'polls.models.Question'>
SELECT "polls_question"."id", "polls_question"."question_text", "polls_question"."pub_date" FROM "polls_question"


In [5]:
for question in result:
    print(question.pk, question.question_text, question.pub_date)

1 좋아하는 색은 무엇입니까? 2026-07-24 06:39:36.233670+00:00
2 싫어하는 색은 무엇입니까? 2026-07-24 06:40:25.595851+00:00
3 좋아하는 동물은 무엇입니까? 2026-07-24 06:40:35.078793+00:00
4 여행가고 싶은 나라는 어디인가요? 2026-07-24 06:40:51.811300+00:00


In [ ]:
result[:2] # list

[<Question: 1. 좋아하는 색은 무엇입니까?>, <Question: 2. 싫어하는 색은 무엇입니까?>]

In [ ]:
# result[1]
# result[-1]  # 음수 indexing은 지원안함.

ValueError: Negative indexing is not supported.

In [ ]:
result.first() # 첫번째 값
result.last()  # 마지막 값

<Question: 4. 여행가고 싶은 나라는 어디인가요?>

In [ ]:
Question.objects.get(pk=1) # where pk = 1

<Question: 1. 좋아하는 색은 무엇입니까?>

In [25]:
# Choice.objects.get(vote=0)   # 조회결과가 여러개인 경우는 에러발생.
try:
    Choice.objects.get(vote=300000)# 조회결과가 없는 경우 에러발생
except:
    print("조회결과가 없습니다.")

조회결과가 없습니다.


In [27]:
# 조회결과가 여러개인 경우
result = Choice.objects.filter(vote=0) # QuerySet
len(result)
result

<QuerySet [<Choice: 2. 검정색>, <Choice: 3. 보라색>, <Choice: 4. 빨강색>, <Choice: 5. 파랑색>, <Choice: 6. 검정색>, <Choice: 7. 주황색>, <Choice: 8. 빨강색>, <Choice: 9. 호랑이>, <Choice: 10. 개>, <Choice: 17. 일본>]>

In [29]:
result[0].pk, result[0].choice_text, result[0].vote

(2, '검정색', 0)

In [30]:
result[0].question

<Question: 1. 좋아하는 색은 무엇입니까?>

In [31]:
result = Choice.objects.exclude(vote=0) # where not vote=0
len(result)

8

In [32]:
for c in result:
    print(c.choice_text, c.vote)

파랑색 12
사자 60
고양이 122
원숭이 32
영국 20
미국 123
프랑스 33
중국 15


In [53]:
# Where 조건
# Field명__연산자 = 비교할값
result = Choice.objects.filter(vote=0) # where vote = 0
result = Choice.objects.filter(vote__lt=50) # where vote < 50
result = Choice.objects.filter(vote__lte=33) # vote <= 33
result = Choice.objects.filter(vote__gt=60)   # vote > 60
result = Choice.objects.filter(vote__gte=60)   # vote >= 60

# 문자열
result = Choice.objects.filter(choice_text="빨강색")   
result = Choice.objects.filter(choice_text__startswith="파랑") # choice_text like '파랑%'
result = Choice.objects.filter(choice_text__endswith="색") # choice_text like '%색'
result = Choice.objects.filter(choice_text__contains="라") # choice_text like '%라%'

result = Choice.objects.filter(choice_text__in = ['보라색', '빨강색', "개"])
result = Choice.objects.filter(vote__range=[50, 100])# vote between 50 and 100

for choice in result:
    print(choice.choice_text, choice.vote)

print(result.query) #QuerySet.query : 실행된 SQL문

사자 60
SELECT "polls_choice"."id", "polls_choice"."choice_text", "polls_choice"."vote", "polls_choice"."question_id" FROM "polls_choice" WHERE "polls_choice"."vote" BETWEEN 50 AND 100


In [ ]:
# 조건이 여러개인 경우 - AND, OR
# AND - 조건을 나열.
result = Choice.objects.filter(
    vote__lt = 100,
    choice_text__in = ['파랑색', '검정색', '개', '고양이']
)

print(result.query)
for choice in result:
    print(choice.choice_text, choice.vote)


SELECT "polls_choice"."id", "polls_choice"."choice_text", "polls_choice"."vote", "polls_choice"."question_id" FROM "polls_choice" WHERE ("polls_choice"."choice_text" IN (파랑색, 검정색, 개, 고양이) AND "polls_choice"."vote" < 100)
파랑색 12
검정색 0
파랑색 0
검정색 0
개 0


In [ ]:
# OR - Q 클래스에 조건을 넣어 주고 `|`  연산으로 묶어준다.
## Q(조건) | Q(조건) | Q(조건)
from django.db.models import Q 
# vote 가 10 이하거나 100 이상

# ~Q(조건)  not 조건
result = Choice.objects.filter(
    Q(vote__lte=10) | Q(vote__gte=100)
)
print(result.query)
for choice in result:
    print(choice.choice_text, choice.vote)

SELECT "polls_choice"."id", "polls_choice"."choice_text", "polls_choice"."vote", "polls_choice"."question_id" FROM "polls_choice" WHERE ("polls_choice"."vote" <= 10 OR "polls_choice"."vote" >= 100)
검정색 0
보라색 0
빨강색 0
파랑색 0
검정색 0
주황색 0
빨강색 0
호랑이 0
개 0
고양이 122
미국 123
일본 0


In [63]:
# 결과 정렬 - order_by("기준 컬럼"): ASC 정렬, DESC: "-기준 컬럼"
## vote의 오름차순
result = Choice.objects.all().order_by("vote")
## 내림차순
result = Choice.objects.all().order_by("-vote")
## 2(n)차 정렬
### vote: DESC, choice_text: ASC
result = Choice.objects.all().order_by("-vote", "choice_text")
print(result.query)
for choice in result:
    print(choice.choice_text, choice.vote)

SELECT "polls_choice"."id", "polls_choice"."choice_text", "polls_choice"."vote", "polls_choice"."question_id" FROM "polls_choice" ORDER BY "polls_choice"."vote" DESC, "polls_choice"."choice_text" ASC
미국 123
고양이 122
사자 60
프랑스 33
원숭이 32
영국 20
중국 15
파랑색 12
개 0
검정색 0
검정색 0
보라색 0
빨강색 0
빨강색 0
일본 0
주황색 0
파랑색 0
호랑이 0


In [69]:
# 특정 컬럼들만 지정해서 조회. select 컬럼명

result = Choice.objects.all().values("pk", "choice_text")
print(type(result)) # QuerySet
print(result.query)
print(type(result[0])) # 개별결과: dict

for value in result:
    print(value['pk'], value['choice_text'], value)

<class 'django.db.models.query.QuerySet'>
SELECT "polls_choice"."id" AS "pk", "polls_choice"."choice_text" AS "choice_text" FROM "polls_choice"
<class 'dict'>
1 파랑색 {'pk': 1, 'choice_text': '파랑색'}
2 검정색 {'pk': 2, 'choice_text': '검정색'}
3 보라색 {'pk': 3, 'choice_text': '보라색'}
4 빨강색 {'pk': 4, 'choice_text': '빨강색'}
5 파랑색 {'pk': 5, 'choice_text': '파랑색'}
6 검정색 {'pk': 6, 'choice_text': '검정색'}
7 주황색 {'pk': 7, 'choice_text': '주황색'}
8 빨강색 {'pk': 8, 'choice_text': '빨강색'}
9 호랑이 {'pk': 9, 'choice_text': '호랑이'}
10 개 {'pk': 10, 'choice_text': '개'}
11 사자 {'pk': 11, 'choice_text': '사자'}
12 고양이 {'pk': 12, 'choice_text': '고양이'}
13 원숭이 {'pk': 13, 'choice_text': '원숭이'}
14 영국 {'pk': 14, 'choice_text': '영국'}
15 미국 {'pk': 15, 'choice_text': '미국'}
16 프랑스 {'pk': 16, 'choice_text': '프랑스'}
17 일본 {'pk': 17, 'choice_text': '일본'}
18 중국 {'pk': 18, 'choice_text': '중국'}


# 집계
- `aggregate(집계함수("Field명"), 집계함수("Field명"), ...)`
    - 전체 집계
- `values("groupby_기준컬럼").annotate(집계함수("필드명), ...)`
    - group by 후 집계

In [9]:
from polls.models import Question, Choice
from django.db import models

In [11]:
# Choice의 개수, vote의 평균
result = Choice.objects.aggregate(
    models.Count("id"), # key: field명__집계
    models.Avg("vote")
)
print(result)

{'id__count': 18, 'vote__avg': 23.166666666666668}


In [12]:
result = Choice.objects.aggregate(
    cnt=models.Count("id"), # keyword argument로 전달. 변수명이 key가됨.
    avg=models.Avg("vote")
)
print(result)

{'cnt': 18, 'avg': 23.166666666666668}


In [15]:
result = Choice.objects.aggregate(
    models.Min("vote"), 
    models.Max("vote"),
    models.Sum("vote"),
    models.StdDev("vote"),
    models.Variance("vote")
)
result

{'vote__min': 0,
 'vote__max': 123,
 'vote__sum': 417,
 'vote__stddev': 38.61095123867781,
 'vote__variance': 1490.8055555555557}

In [16]:
result['vote__sum']

417

In [22]:
# group by
## select min(vote), max(vote) from choice group by question
result = Choice.objects.values("question").annotate(
    models.Min("vote"), models.Max("vote")
)
print(type(result))
print(result)

<class 'django.db.models.query.QuerySet'>
<QuerySet [{'question': 1, 'vote__min': 0, 'vote__max': 12}, {'question': 2, 'vote__min': 0, 'vote__max': 0}, {'question': 3, 'vote__min': 0, 'vote__max': 122}, {'question': 4, 'vote__min': 0, 'vote__max': 123}]>


In [23]:
for r in result:
    print(r)

{'question': 1, 'vote__min': 0, 'vote__max': 12}
{'question': 2, 'vote__min': 0, 'vote__max': 0}
{'question': 3, 'vote__min': 0, 'vote__max': 122}
{'question': 4, 'vote__min': 0, 'vote__max': 123}


# JOIN

In [27]:
##############################################
# 자식기준 조회 -> 참조하는 부모 행도 같이 조회
##############################################
c1 = Choice.objects.get(pk=1)
q1 = c1.question
print(c1.pk, c1.choice_text, c1.vote)
print(c1.question, type(c1.question))

1 파랑색 12
1. 좋아하는 색은 무엇입니까? <class 'polls.models.Question'>


In [28]:
q1.question_text, c1.choice_text

('좋아하는 색은 무엇입니까?', '파랑색')

In [30]:
# 보기글에 "색" 을 포함 것들.
# 출력 - 보기의 질문내용, 보기내용
result = Choice.objects.filter(choice_text__contains="색")
for choice in result:
    print(choice.choice_text, choice.question.question_text, sep=", ")

파랑색, 좋아하는 색은 무엇입니까?
검정색, 좋아하는 색은 무엇입니까?
보라색, 좋아하는 색은 무엇입니까?
빨강색, 좋아하는 색은 무엇입니까?
파랑색, 싫어하는 색은 무엇입니까?
검정색, 싫어하는 색은 무엇입니까?
주황색, 싫어하는 색은 무엇입니까?
빨강색, 싫어하는 색은 무엇입니까?


In [ ]:
############################################################
# 부모모델(테이블)을 조회하면서 자식모델을 join해서 조회할 경우
#
# 조회된_부모_모델객체.related_name : 
#  - 부모모델객체를 참조하는 자식 행들을 조회할 수 있는 (Model) Manager를 반환
#  - related_name패턴: 자식모델클래스이름(소문자)_set (default 이름)
#                      ForeignKey(..., related_name="relatedName지정")
############################################################

In [33]:
q1 = Question.objects.get(pk=1)
q1.pk, q1.question_text, q1.pub_date.strftime("%Y-%m-%d")

(1, '좋아하는 색은 무엇입니까?', '2026-07-24')

In [36]:
# q1의 choice(보기)들 조회
### choice_set은 q1을 참조하는 choice들을 조회대상으로하는 manager
c = q1.choice_set.all() 
len(c)

4

In [38]:
print("질문:", q1.pk, q1.question_text)
print("보기")
for choice in c:
    print(choice.pk, choice.choice_text)

질문: 1 좋아하는 색은 무엇입니까?
보기
1 파랑색
2 검정색
3 보라색
4 빨강색


# Insert/Update
- `Model객체.save()`
    - 모델객체를 한행으로 insert/update 처리 
    - instance 변수들의 값이 연결된 테이블의 컬럼에 저장.
- 모델객체의 `pk`가 테이블에 없으면 `insert`, 있으면 `update`

In [41]:
# insert
new_question = Question(question_text="싫어하는 운동은 무엇입니까?")
# pk, pub_date => None (insert할때 자동으로 등록)
new_question.save()

In [42]:
qs = Question.objects.all()
for q in qs:
    print(q)

1. 좋아하는 색은 무엇입니까?
2. 싫어하는 색은 무엇입니까?
3. 좋아하는 동물은 무엇입니까?
4. 여행가고 싶은 나라는 어디인가요?
6. 좋아하는 운동은 무엇입니까?
7. 싫어하는 운동은 무엇입니까?


In [44]:
# 업데이트
update_question = Question.objects.get(pk=6)
print(update_question)
print(update_question.pub_date)

6. 좋아하는 운동은 무엇입니까?
2026-07-27 02:27:58.655047+00:00


In [45]:
update_question.question_text = "무슨 운동을 좋아해요?"
update_question.save() 

In [46]:
r = Question.objects.get(pk=6)
print(r)
print(r.pub_date)

6. 무슨 운동을 좋아해요?
2026-07-27 02:27:58.655047+00:00


# Delete
- `모델객체.delete()`
- 모델의 PK속성의 행을 삭제

In [48]:
del_q = Question(pk=7) # question_text, pub_date: None
del_q.delete() # delete from question where id=model.pk

(1, {'polls.Question': 1})

In [52]:
# 삭제할 대상들을 조회
del_qs = Question.objects.filter(pk__in=[2, 6])
# 조회된 대상들을 삭제
for q in del_qs:
    q.delete() 

In [53]:
qs = Question.objects.all()

for q in qs:
    print(q)

1. 좋아하는 색은 무엇입니까?
3. 좋아하는 동물은 무엇입니까?
4. 여행가고 싶은 나라는 어디인가요?
